# Simple English RAG (`ser`) — High-Speed GPU Ingestion on Google Colab

This notebook uses Google Colab's free **NVIDIA T4 GPU** to stream, chunk, embed, and ingest the Simple English Wikipedia dump directly into your hosted **Qdrant Cloud** cluster at **~400–600 chunks/second**.

### Quick Setup (3 Steps):
1. In the Colab menu above, ensure GPU is enabled: **Runtime** > **Change runtime type** > select **T4 GPU**.
2. In **Step 2**, enter your `QDRANT_API_KEY`.
3. Click **Runtime** > **Run all** (or press `Ctrl + F9`)!

In [ ]:
# Step 1: Clone or pull latest code and install dependencies
import os, sys
from pathlib import Path

if os.path.exists("/content/SimpleEnglishRag"): 
    %cd /content/SimpleEnglishRag
    !git pull origin main
else:
    %cd /content
    !git clone https://github.com/parm2006/SimpleEnglishRag.git
    %cd /content/SimpleEnglishRag

# Ensure src/ is immediately accessible to Python
src_path = str(Path("/content/SimpleEnglishRag/src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Install core dependencies with GPU support
!pip install -q --upgrade pip
!pip install -q onnxruntime-gpu fastembed-gpu qdrant-client rich python-dotenv httpx
!pip install -q -e . --no-deps

# Verify GPU acceleration
import onnxruntime as ort
providers = ort.get_available_providers()
print("Available ONNX Execution Providers:", providers)
if "CUDAExecutionProvider" in providers:
    print("\n SUCCESS: NVIDIA CUDA GPU is active and ready for high-speed embeddings!")
else:
    print("\n WARNING: CUDA not detected. Switch runtime: Runtime > Change runtime type > T4 GPU")

In [ ]:
# Step 2: Configure Qdrant Cloud credentials
import os

# Attempt to read from Colab Secrets
url_secret = None
key_secret = None
try:
    from google.colab import userdata
    url_secret = userdata.get('QDRANT_URL')
    key_secret = userdata.get('QDRANT_API_KEY')
except Exception:
    pass

# Set your Qdrant credentials
QDRANT_URL = "https://your-cluster-id.cloud.qdrant.io" #@param {type:"string"}
QDRANT_API_KEY = "" #@param {type:"string"}
COLLECTION_NAME = "simple_wiki" #@param {type:"string"}

active_url = (url_secret or QDRANT_URL).strip()
active_key = (key_secret or QDRANT_API_KEY).strip()

if not active_url or "your-cluster-id" in active_url:
    raise ValueError("Please enter your QDRANT_URL in the field above or in Colab Secrets (🔑)!")
if not active_key:
    raise ValueError("Please enter your QDRANT_API_KEY in the field above or in Colab Secrets (🔑)!")

os.environ["QDRANT_STORAGE"] = "cloud"
os.environ["QDRANT_COLLECTION"] = COLLECTION_NAME
os.environ["QDRANT_URL"] = active_url
os.environ["QDRANT_API_KEY"] = active_key

from ser.db import client, COLLECTION_NAME
info = client.get_collection(COLLECTION_NAME)
print(f" Connected to Qdrant Cloud! Currently holds {info.points_count:,} vectors in status '{info.status}'.")

In [ ]:
# Step 3: Stream-download the official Simple English Wikipedia dump (~5-10s in Google Cloud)
from ser.download import download_dump
dump_file = download_dump(dest_dir="data")
print(" Dump file ready at:", dump_file)

In [ ]:
# Step 4: Resume GPU Ingestion
import json, time
from pathlib import Path
from fastembed import TextEmbedding
from ser.dump import iter_dump_articles
from ser.pipeline import index_documents
import ser.embed

# 1. Initialize FastEmbed to use the NVIDIA CUDA GPU
print("Initializing FastEmbed ONNX on GPU...")
ser.embed.model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    providers=["CUDAExecutionProvider"]
)

# 2. Set checkpoint to resume from article 2,700 (skip already-indexed articles)
checkpoint_path = Path("data/ingest_checkpoint.json")
if not checkpoint_path.exists():
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    with open(checkpoint_path, "w", encoding="utf-8") as f:
        json.dump({"last_page_id": "8646", "total_processed": 2700}, f)
    print(" Resuming from article 2,700 (Page ID 8646). Zero duplicate work!")

# 3. Stream and ingest remaining articles
dump_path = Path("data/simplewiki-latest-pages-articles.xml.bz2")
articles_stream = iter_dump_articles(
    dump_path,
    checkpoint_path=checkpoint_path,
    reset=False
)

print("\n Launching GPU ingestion into Qdrant Cloud (batch_size=256)...\n")
t0 = time.time()
total = index_documents(articles_stream, client=client, batch_size=256)
t1 = time.time()

print(f"\n Ingestion Complete! Added {total:,} chunks in {(t1-t0)/60:.1f} minutes.")

In [ ]:
# Step 5: Final Collection Summary & Live Search Test
from ser.pipeline import ask

info = client.get_collection(COLLECTION_NAME)
print(f" Qdrant Cloud Collection: {COLLECTION_NAME}")
print(f" Total Vectors in Cloud: {info.points_count:,}")
print(f" Status: {info.status}")

# Quick sanity query
test_query = "What is artificial intelligence?"
print(f"\n Testing Search: '{test_query}'")
hits = ask(client, test_query, k=2)
for h in hits:
    print(f"- [{h.score:.4f}] {h.payload.get('title')}: {h.payload.get('text')[:120]}...")